# **B-cos Explanations**

**Authors:** Katrine Bjerre (katbj@itu.dk) & Kristine Emilie Risager Pedersen (krep@itu.dk)

Last edited: 28.04.2026


## **Table of Contents**

1. [Imports & Setup](#imports--setup)
2. [Load Model Configuration](#load-model-configuration)
3. [Helper Functions](#Helper-functions:-data,-paths,-model-loading)
4. [Image Preprocessing](#image-preprocessing-to-match-model-input)
5. [B-cos Explanation Implementation](#b-cos-explanation-implementation)
6. [Plot Helpers](#plot-helpers)
7. [Single Run Helper](#single-run-helper)
8. [Run B-cos Explanations](#run-b-cos-explanations)


## **Imports & Setup**

In [1]:
from pathlib import Path
import json
import sys
import cv2
import matplotlib.pyplot as plt
from matplotlib import cm
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from PIL import Image
from torchvision import transforms
import bcos

In [2]:
plt.rcParams.update({
    "text.usetex": False,
    "font.family": "serif",
    "mathtext.fontset": "cm",
    "font.size": 14,
    "axes.labelsize": 15,
    "xtick.labelsize": 13,
    "ytick.labelsize": 13,
})

In [ ]:
def resolve_project_root() -> Path:
    """Find project root directory."""
    cwd = Path.cwd().resolve()

    if (cwd / "data_utils").exists() and (cwd / "results").exists():
        return cwd

    if cwd.name == "notebooks" and (cwd.parent / "data_utils").exists():
        return cwd.parent

    raise RuntimeError("Could not resolve project root.")


PROJECT_ROOT = resolve_project_root()

# Add project root to Python path for imports
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from data_utils.dataset import load_metadata
from data_utils.subject_groups import get_person_specific_subjects
from models.bcos_model import build_bcos_model

In [4]:
DEVICE = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)

print("Device:", DEVICE)

Device: mps


In [5]:
# Paths to data, config, and trained models
METADATA_PATH = PROJECT_ROOT / "data_utils" / "metadata.csv"
CONFIG_PATH = PROJECT_ROOT / "results" / "grid_search" / "best_config.json"
MODEL_BASE_DIR = PROJECT_ROOT / "results" / "models"

# Output directories for each model type
OUTPUT_ROOT = PROJECT_ROOT / "results" / "b_cos"
OUTPUT_DIRS = {
    "generalized": OUTPUT_ROOT / "generalized_bcos",
    "subject_spec_scratch": OUTPUT_ROOT / "person_specific_scratch_bcos",
    "subject_spec_transfer_last_two_conv": OUTPUT_ROOT / "person_specific_transfer_last_two_conv_bcos",
}

# Create output folders if they do not exist
for p in OUTPUT_DIRS.values():
    p.mkdir(parents=True, exist_ok=True)

# Evaluation setup
HELD_OUT_SUBJECTS = get_person_specific_subjects()
NUM_TARGETS = 25
TARGETS = list(range(NUM_TARGETS))

print("Held-out subjects:", HELD_OUT_SUBJECTS)
print("Targets:", TARGETS)

# Image and saving settings
IMAGE_SIZE = (96, 128)
JPEG_DPI = 200

# Model variants to evaluate
MODEL_TYPES = [
    "generalized",
    "subject_spec_scratch",
    "subject_spec_transfer_last_two_conv",
]

Held-out subjects: ['002', '007', '023', '034', '037']
Targets: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24]


## **Load Model Configuration**

In [6]:
# Load best model configuration from JSON
with open(CONFIG_PATH, "r") as f:
    config_file = json.load(f)

# Extract relevant parameters for model construction
raw_cfg = config_file["best_config"]

# Final config used to build the B-cos model
MODEL_CONFIG = {
    "num_blocks": raw_cfg["num_blocks"],
    "base_channel": raw_cfg["base_channel"],
    "adaptive_pool_out": tuple(raw_cfg["adaptive_pool_out"]),
    "hidden_dim": raw_cfg["hidden_dim"],
    "activation": raw_cfg["activation"],
    "b": 2.0,
    "max_out": 1,
    "use_bcos_linear": False,
}

print("MODEL_CONFIG:", MODEL_CONFIG)

MODEL_CONFIG: {'num_blocks': 4, 'base_channel': 32, 'adaptive_pool_out': (3, 4), 'hidden_dim': 32, 'activation': 'leakyrelu', 'b': 2.0, 'max_out': 1, 'use_bcos_linear': False}


## **Helper Functions**

In [7]:
def zfill_subject(subject) -> str:
    """Format subject ID as zero-padded string (e.g., 2 → '002')."""
    return str(subject).zfill(3)


def get_checkpoint_path(model_type: str, subject: str, target: int) -> Path:
    """Return path to trained model checkpoint."""
    subject = zfill_subject(subject)
    target = int(target)

    filenames = {
        "generalized": f"gen_loto_bcos_{target}.pth",
        "subject_spec_scratch": f"subject_spec_scratch_bcos_{subject}_{target}.pth",
        "subject_spec_transfer_last_two_conv": f"subject_spec_transfer_last_two_conv_{subject}_{target}_bcos.pth",
    }

    if model_type not in filenames:
        raise ValueError(f"Unknown model_type: {model_type}")

    checkpoint_path = MODEL_BASE_DIR / filenames[model_type]

    # Ensure checkpoint exists
    if not checkpoint_path.exists():
        raise FileNotFoundError(f"Checkpoint not found: {checkpoint_path}")

    return checkpoint_path


def load_trained_model(checkpoint_path: Path, model_config: dict, device=DEVICE):
    """Build model and load trained weights."""
    model = build_bcos_model(**model_config).to(device)
    checkpoint = torch.load(checkpoint_path, map_location=device)

    model.load_state_dict(checkpoint)
    model.eval()
    return model


def get_output_path(model_type: str, subject: str, target: int) -> Path:
    """Return path for saving output figure."""
    subject = zfill_subject(subject)
    target = int(target)
    return OUTPUT_DIRS[model_type] / f"{subject}_{target}.png"


def select_sample(df: pd.DataFrame, subject: str, target: int):
    """Select one sample for given subject and target."""
    subject = zfill_subject(subject)
    target = int(target)

    subset = df[(df["subject"] == subject) & (df["target"] == target)].copy()

    if subset.empty:
        raise ValueError(f"No sample found for subject={subject}, target={target}")

    return subset.sort_values("path").iloc[0]


def extract_target_coords(row: pd.Series):
    """Extract ground truth coordinates."""
    return float(row["x_pixel"]), float(row["y_pixel"])

## **Image Preprocessing**

In [8]:
# Image preprocessing pipeline
_transform = transforms.Compose([
    transforms.Resize(IMAGE_SIZE),
    transforms.ToTensor(),
])


def load_image(image_path: Path):
    """Load image and return numpy + model input tensor."""
    
    # Check that image exists
    if not image_path.exists():
        raise FileNotFoundError(f"Image not found: {image_path}")

    # Load as grayscale image
    pil_img = Image.open(image_path).convert("L")

    # Keep original image for visualization
    original_np = np.array(pil_img)

    # Transform to tensor with shape [1, 1, H, W]
    input_tensor = _transform(pil_img).unsqueeze(0).float()

    return original_np, input_tensor

## **B-cos Explanation Implementation**

In [9]:
def explain_bcos_regression_native(
    model: nn.Module,
    input_tensor: torch.Tensor,
    output_idx: int,
    device: torch.device,
):
    """Compute B-cos contribution map for one output dimension."""
    
    model = model.to(device)
    model.eval()

    # Prepare input with gradients enabled
    x = input_tensor.clone().detach().to(device)
    x.requires_grad_(True)

    model.zero_grad(set_to_none=True)

    # Forward + backward pass in B-cos explanation mode
    with bcos.explanation_mode(model):
        pred = model(x)
        scalar = pred[:, output_idx].sum()
        scalar.backward()

    # Contribution per pixel (input * gradient)
    contribution_map = (x.detach() * x.grad.detach()).sum(dim=1)
    contribution_map = contribution_map.detach().cpu().numpy()[0]

    # Model prediction
    pred_np = pred.detach().cpu().numpy()[0].astype(np.float32)

    return contribution_map, pred_np

## **Plot Helpers**

In [ ]:
def normalize_importance_map(arr: np.ndarray, percentile: float = 99.5) -> np.ndarray:
    """Normalize contribution map to [0, 1]."""
    
    # Use absolute values (ignore sign)
    arr = np.abs(arr)

    # Scale using percentile to reduce effect of outliers
    scale = np.percentile(arr, percentile)

    if scale <= 1e-12:
        return np.zeros_like(arr, dtype=np.float32)

    # Normalize and clip to [0, 1]
    arr = np.clip(arr / scale, 0.0, 1.0)
    return arr.astype(np.float32)


def resize_map(arr: np.ndarray, target_shape):
    """Resize map to match image size."""
    h, w = target_shape
    return cv2.resize(arr, (w, h), interpolation=cv2.INTER_CUBIC)


def make_importance_overlay(
    image: np.ndarray,
    importance_map: np.ndarray,
    alpha: float = 0.7,
    threshold: float = 0.0,
    cmap_name: str = "jet",
):
    """Overlay importance map on image."""

    # Remove low-importance regions
    strength = importance_map.copy()
    strength[strength < threshold] = 0.0

    # Convert to color heatmap
    rgb_heat = cm.get_cmap(cmap_name)(strength)[..., :3]

    # Convert grayscale to RGB
    base = cv2.cvtColor(image, cv2.COLOR_GRAY2RGB).astype(np.float32) / 255.0

    # Blend heatmap with image
    alpha_map = alpha * strength[..., None]
    overlay = (1.0 - alpha_map) * base + alpha_map * rgb_heat

    return np.clip(overlay, 0.0, 1.0)


def plot_bcos_importance(ax, importance_map, title, cmap="jet"):
    """Plot normalized importance map."""

    # Show heatmap
    ax.imshow(
        importance_map,
        cmap=cmap,
        vmin=0.0,
        vmax=1.0,
        interpolation="bilinear"
    )

    ax.set_title(title)
    ax.axis("off")


def save_bcos_figure_native(
    image_np,
    contrib_x,
    contrib_y,
    pred,
    target_coords,
    subject,
    target_idx,
    model_type,
    save_path,
    overlay_alpha: float = 0.80,
    overlay_threshold: float = 0.0,
):
    """Create and save B-cos visualization (image, maps, overlays)."""

    # Match contribution maps to image size
    h, w = image_np.shape[:2]
    contrib_x = resize_map(contrib_x, (h, w))
    contrib_y = resize_map(contrib_y, (h, w))

    # Normalize maps to [0,1]
    importance_x = normalize_importance_map(contrib_x, percentile=99.5)
    importance_y = normalize_importance_map(contrib_y, percentile=99.5)

    # Create overlay images
    overlay_x = make_importance_overlay(
        image_np,
        importance_x,
        alpha=overlay_alpha,
        threshold=overlay_threshold,
        cmap_name="jet",
    )

    overlay_y = make_importance_overlay(
        image_np,
        importance_y,
        alpha=overlay_alpha,
        threshold=overlay_threshold,
        cmap_name="jet",
    )

    # Create figure with 2 rows (x and y)
    fig, axes = plt.subplots(2, 3, figsize=(16, 10), dpi=JPEG_DPI, constrained_layout=True)

    # --- X output ---
    axes[0, 0].imshow(image_np, cmap="gray", interpolation="bilinear")
    axes[0, 0].set_title("Input image")
    axes[0, 0].axis("off")

    plot_bcos_importance(
        axes[0, 1],
        importance_x,
        "B-cos for x",
        cmap="jet",
    )

    axes[0, 2].imshow(overlay_x, interpolation="bilinear")
    axes[0, 2].set_title("Overlay for x")
    axes[0, 2].axis("off")

    # --- Y output ---
    axes[1, 0].imshow(image_np, cmap="gray", interpolation="bilinear")
    axes[1, 0].set_title("Input image")
    axes[1, 0].axis("off")

    plot_bcos_importance(
        axes[1, 1],
        importance_y,
        "B-cos for y",
        cmap="jet",
    )

    axes[1, 2].imshow(overlay_y, interpolation="bilinear")
    axes[1, 2].set_title("Overlay for y")
    axes[1, 2].axis("off")

    # Add title with prediction and ground truth
    fig.suptitle(
        f"B-cos | {model_type} | subject {subject} | target {target_idx}\n"
        f"Pred: ({pred[0]:.3f}, {pred[1]:.3f}) | Target: ({target_coords[0]:.3f}, {target_coords[1]:.3f})",
        fontsize=18,
    )

    # Ensure directory exists and save figure
    save_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(save_path, dpi=JPEG_DPI, bbox_inches="tight")

    plt.close(fig)

## **Single-Run Helper**

In [11]:
def run_and_save_single_native(model_type: str, sample_row: pd.Series):
    """Run BCos explanation for one sample and save visualization."""

    # Extract identifiers
    subject = zfill_subject(sample_row["subject"])
    target = int(sample_row["target"])

    # Load trained model
    checkpoint_path = get_checkpoint_path(model_type, subject, target)
    model = load_trained_model(checkpoint_path, MODEL_CONFIG, device=DEVICE)

    # Resolve image path
    image_path = Path(sample_row["path"])
    if not image_path.is_absolute():
        image_path = PROJECT_ROOT / image_path

    # Load image and ground truth
    image_np, input_tensor = load_image(image_path)
    target_coords = extract_target_coords(sample_row)

    # Compute B-cos explanations for x-coordinate
    contrib_x, pred = explain_bcos_regression_native(
        model=model,
        input_tensor=input_tensor,
        output_idx=0,
        device=DEVICE,
    )

    # Compute B-cos explanations for y-coordinate
    contrib_y, pred2 = explain_bcos_regression_native(
        model=model,
        input_tensor=input_tensor,
        output_idx=1,
        device=DEVICE,
    )

    # Ensure predictions match
    if not np.allclose(pred, pred2, atol=1e-5):
        pred = pred2

    # Save visualization
    save_path = get_output_path(model_type, subject, target)

    save_bcos_figure_native(
        image_np=image_np,
        contrib_x=contrib_x,
        contrib_y=contrib_y,
        pred=pred,
        target_coords=target_coords,
        subject=subject,
        target_idx=target,
        model_type=model_type,
        save_path=save_path,
        overlay_alpha=0.80,
        overlay_threshold=0.0,
    )

    return save_path

## **Run B-cos Explanations**

In [ ]:
# Load metadata
df = load_metadata(METADATA_PATH)

saved = []
failures = []

# Held-out subjects: ['002', '007', '023', '034', '037']
#SUBJECT_TO_RUN = '037'

# Run explanations for all subjects, targets, and model types
for subject in HELD_OUT_SUBJECTS:
    #if subject != SUBJECT_TO_RUN:
        #continue
    for target in TARGETS:

        try:
            sample_row = select_sample(df, subject=subject, target=target)
        except Exception as e:
            failures.append({
                "subject": subject,
                "target": target,
                "stage_or_model": "sample_selection",
                "error": str(e),
            })
            continue

        for model_type in MODEL_TYPES:
            print(f"Running {model_type} | subject={subject} target={target}")

            try:
                save_path = run_and_save_single_native(model_type, sample_row)
                saved.append({
                    "model_type": model_type,
                    "subject": subject,
                    "target": target,
                    "path": str(save_path),
                })
                print("[OK]")

            except Exception as e:
                failures.append({
                    "subject": subject,
                    "target": target,
                    "stage_or_model": model_type,
                    "error": str(e),
                })
                print("[FAIL]")

# Store run summary
saved_df = pd.DataFrame(saved)
failures_df = pd.DataFrame(failures)

# Show failures, if any
if not failures_df.empty:
    display(failures_df.head(20))

Running generalized | subject=037 target=0


/var/folders/4l/cfqj812j3yvfgvxxw_tcfbq80000gn/T/ipykernel_86526/1997073983.py:38: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  rgb_heat = cm.get_cmap(cmap_name)(strength)[..., :3]


[OK]
Running subject_spec_scratch | subject=037 target=0
[OK]
Running subject_spec_transfer_last_two_conv | subject=037 target=0
[OK]
Running generalized | subject=037 target=1
[OK]
Running subject_spec_scratch | subject=037 target=1
[OK]
Running subject_spec_transfer_last_two_conv | subject=037 target=1
[OK]
Running generalized | subject=037 target=2
[OK]
Running subject_spec_scratch | subject=037 target=2
[OK]
Running subject_spec_transfer_last_two_conv | subject=037 target=2
[OK]
Running generalized | subject=037 target=3
[OK]
Running subject_spec_scratch | subject=037 target=3
[OK]
Running subject_spec_transfer_last_two_conv | subject=037 target=3
[OK]
Running generalized | subject=037 target=4
[OK]
Running subject_spec_scratch | subject=037 target=4
[OK]
Running subject_spec_transfer_last_two_conv | subject=037 target=4
[OK]
Running generalized | subject=037 target=5
[OK]
Running subject_spec_scratch | subject=037 target=5
[OK]
Running subject_spec_transfer_last_two_conv | subject